# Costruzione del panel — versione leggera

Stessa logica di `build_panel.ipynb`, **senza le colonne che non servono**.

Il panel completo produce 117 colonne; `data/raw/example_panel.csv` ne usa **53**.
Delle 64 di troppo, nessuna è letta da niente: due fasi intere (3a competitor
statici, 3b dipendenti/bilanci/news) spariscono, e quattro CSV non si aprono più.

**Cosa resta identico:** ogni semantica R riprodotta — `r_if_else`, `r_cum_sum`,
`r_seq`, `scale_r`, `cumany`, `coalesce_first_last` — le imputazioni di
`StartDate`/`EndDate`, `Zero_Invested`, la riparazione delle date dei deal, il
`full_join` della 2b, il troncamento.

**Cosa cambia:** due difetti dell'R sono **corretti in modo permanente**,
non più dietro un flag.

| difetto | dove | effetto della correzione |
|---|---|---|
| **B1** — la fase 1 filtra `YearFounded > 1999`, le fasi 2b e 4 `> 2000` | 1.6, 2b.1, 4.4 | 1.422 aziende della coorte 2000 passano da 0,0% a 90,5% di righe con dati di team |
| **B6** — `seq()` conta all'indietro e genera anni precedenti alla fondazione | 1.4 | 186 righe con `Age < 0` spariscono, nessuna azienda viene persa |

B1 non è corretto «a mano» in tre punti: la soglia è **una costante sola**,
`ANNO_MIN_FONDAZIONE`, letta da `first_year` in `config/config.yaml` e usata in
tutti e tre i filtri. Finché la sorgente è una, le tre fasi non possono più
divergere. È inclusiva: `first_year: 2000` ammette le aziende fondate nel 2000.

Entrambe sono state verificate contro il notebook completo eseguito con gli
stessi due flag accesi: **zero divergenze su tutte e 51 le colonne condivise**.
Gli altri flag `fix_*` restano spenti e riproducono il difetto dell'R.

Gli output vanno in `data/interim_light/`, separati da quelli del panel completo.

In [ ]:
# ── Import ────────────────────────────────────────────────────────────────
import dataclasses                    # serve solo a clonare PanelConfig cambiandone un campo
import gc                             # garbage collection manuale fra uno stadio e l'altro
import sys
from pathlib import Path

import polars as pl
import yaml

ROOT = Path.cwd()                     # la cartella da cui gira il notebook = radice del repo
if str(ROOT) not in sys.path:         # senza questo, "from src.panel..." non trova i moduli
    sys.path.insert(0, str(ROOT))

from src.panel.config import BUG_FLAGS, PanelConfig
from src.panel.expansions import active_pairs, expand_team
from src.panel.io import COMPANY_DATE_COLUMNS, read_raw, to_num
from src.panel.rutils import (
    R_NA,                  # ("", "NA", "N/A", "NULL")        -> token "mancante" dell'R
    R_NA_INF,              # i precedenti + "NaN", "Inf", "-Inf"
    R_NA_NAN,              # i primi + "NaN"
    as_na,                 # applica quei token a tutte le colonne di un frame
    coalesce_first_last,   # coalesce(first(x), last(x)) della deduplica
    cumany,                # cumany() di dplyr: una volta acceso, resta acceso
    next_different,        # primo valore futuro diverso dal corrente, e a che distanza
    parse_date_r,          # il parser di date a due rami dell'R
    r_case_when,           # catena di grepl a corto circuito: il primo che matcha vince
    r_cum_sum,             # cumsum() dell'R: un NA avvelena tutto il resto del gruppo
    r_if_else,             # if_else() di dplyr: condizione NA -> risultato NA
    r_seq,                 # seq(): inclusivo, e conta ALL'INDIETRO se la fine precede l'inizio
    scale_r,               # scale(): deviazione standard campionaria, denominatore n-1
    tail_na_omit,          # tail(na.omit(x), 1): l'ultimo valore non nullo
    weighted_cumulative,   # media ponderata cumulata
)

# interim_dir separata: il confronto finale ha bisogno che i parquet del panel
# completo, in data/interim/, restino intatti.
cfg = dataclasses.replace(PanelConfig(), interim_dir=Path("data/interim_light"))

pl.Config.set_tbl_cols(12)            # quante colonne mostrare quando si stampa un frame
pl.Config.set_fmt_str_lengths(40)

# ── La soglia di inclusione, letta dalla configurazione ───────────────────
# ANNO_MIN_FONDAZIONE e' l'anno di fondazione piu' antico ammesso nel panel,
# ed e' INCLUSIVO: con 2000, un'azienda fondata nel 2000 entra. Vive in
# config/config.yaml perche' e' una scelta di campione, non un dettaglio
# implementativo, e va usata in tutti e tre i punti che filtrano le aziende
# (fase 1, fase 2b, fase 4). Nell'R quei tre punti usavano due valori diversi:
# e' il difetto B1, corretto qui dal fatto stesso che la costante e' una sola.
with open(ROOT / "config" / "config.yaml") as f:
    CONFIG = yaml.safe_load(f)
ANNO_MIN_FONDAZIONE = int(CONFIG["first_year"])

# Da non confondere con la finestra temporale dei DATASET, che vive a valle in
# src/preprocessing.py e riguarda gli anni di calendario, non di fondazione.

# Due correzioni sono PERMANENTI e non hanno piu' un flag: la soglia unica
# sull'anno di fondazione (B1) e gli anni-azienda negativi (B6). I flag
# restanti valgono ancora False, cioe' riproducono il difetto dell'R.
print(f"anno di fondazione minimo: {ANNO_MIN_FONDAZIONE} (incluso), da config.yaml")
print("correzioni permanenti    : soglia unica in tutte le fasi (B1), "
      "niente anni prima della fondazione (B6)")
attivi = cfg.active_fixes()
print("altri flag accesi        :", list(attivi) if attivi else "nessuno (comportamento R)")
print("output di stadio         :", cfg.interim_dir)

---
## Fase 1 — aziende e scheletro del panel

Da una riga per azienda a **una riga per azienda-anno**. È qui che nasce la
forma del panel.

Di 39 colonne di `Company.csv` ne servono **11**, e quattro di queste non
producono nessuna colonna finale: servono solo a calcolare `MaxYear`, cioè
dove finisce il panel di ogni azienda. Toglierne una sola costa righe —
misurato: senza `FiscalPeriod` se ne perdono 133.942.

In [ ]:
# ── 1.1 · Company.csv, undici colonne su trentanove ───────────────────────
COLONNE_COMPANY = [
    "CompanyID",                    # chiave di tutto il panel
    "YearFounded",                  # anno zero di ogni azienda; e' una delle 53
    "HQCountry",                    # feature
    "PrimaryIndustrySector",        # feature
    "OwnershipStatus",              # NON e' una feature, ma serve in tre punti scollegati:
                                    #   2a -> aziende fallite, per imputare EndDate
                                    #   4  -> riparazione delle date dei deal
                                    #   5  -> i primi tre rami della cascata GrowthStage
    "OwnershipStatusDate",          # idem, piu' MaxYear
    # Le quattro che seguono, piu' FiscalPeriod, non producono nessuna colonna
    # del panel: entrano solo nel pmax che fa MaxYear.
    "CompanyFinancingStatusDate",
    "BusinessStatusDate",
    "FirstFinancingDate",
    "LastKnownValuationDate",
    "FiscalPeriod",                 # "TTM 2Q2019" -> FiscalDate, la sesta data di MaxYear
]

# read_raw legge tutto come stringa e non inferisce i tipi: CompanyID in certi
# file sembra numerico, e un cast implicito romperebbe i join in silenzio.
azienda = read_raw(cfg, "Company", COLONNE_COMPANY)

# as_na riproduce il ciclo dell'R che scrive NA al posto di "", "NA", "N/A",
# "NULL" e "NaN". Va fatto PRIMA di qualunque altra cosa, perche' tutto il
# resto dipende da cosa conta come mancante.
azienda = as_na(azienda, R_NA_NAN)

print(f"{azienda.height:,} aziende x {azienda.width} colonne")

In [ ]:
# ── 1.2 · le date, e FiscalDate ───────────────────────────────────────────
# parse_date_r riproduce la regola a due rami dell'R:
#   10 caratteri -> "%m/%d/%Y"
#    8 caratteri -> mese/giorno/anno a due cifre, con 00-24 nel 2000 e 25-99 nel 1900
#   qualunque altra lunghezza -> NA.
# Su questa estrazione tutte le date hanno 10 caratteri, quindi il secondo ramo
# non scatta mai: e' latente, non attivo.
azienda = azienda.with_columns(
    *[parse_date_r(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    # YearFounded arriva come stringa; strict=False manda a null cio' che non e' un numero.
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

# FiscalPeriod ha la forma "TTM 2Q2019": si estrae il numero del trimestre...
trimestre = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
# ...e l'anno, che sono le ultime quattro cifre della stringa.
anno_fiscale = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
# Il trimestre diventa il mese di chiusura (1Q->3, 2Q->6, 3Q->9, 4Q->12) e il
# giorno e' sempre 30, esattamente come fa l'R con paste(Year, Month, "30").
azienda = azienda.with_columns(pl.date(anno_fiscale, trimestre * 3, 30).alias("FiscalDate"))

print(f"FiscalDate valorizzata su {azienda['FiscalDate'].is_not_null().sum():,} aziende")

In [ ]:
# ── 1.3 · db1: CompanyID e YearFounded, NON filtrati ──────────────────────
# La fase 2a prendera' YearFounded da QUI e non da db_master_1. La differenza
# non e' cosmetica: db_master_1 e' filtrato per anno di fondazione, e usarlo
# farebbe sparire le persone delle aziende piu' vecchie, cambiando db3.
db1 = azienda.select("CompanyID", "YearFounded")
print(f"db1: {db1.height:,} aziende, senza nessun filtro sull'anno di fondazione")

In [ ]:
# ── 1.4 · MaxYear e lo scheletro ──────────────────────────────────────────
# MaxYear = l'anno piu' recente fra le sei date disponibili, cioe' l'ultimo
# anno in cui PitchBook ha un dato su quell'azienda. E' li' che finisce il suo panel.
DATE_MAXYEAR = [*COMPANY_DATE_COLUMNS, "FiscalDate"]

vita = (
    azienda
    # max_horizontal ignora i null: e' il pmax(..., na.rm = TRUE) dell'R.
    # Vale null solo se mancano tutte e sei le date.
    .with_columns(pl.max_horizontal([pl.col(c).dt.year() for c in DATE_MAXYEAR]).alias("MaxYear"))
    # Senza anno di fondazione o senza MaxYear l'azienda non entra nel panel:
    # e' questo il filtro che decide chi c'e' e chi no.
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
    .select("CompanyID", "YearFounded", "MaxYear")
)
print(f"aziende con anno di fondazione e MaxYear: {vita.height:,} su {azienda.height:,}")

# CORREZIONE B6, permanente. Nell'R, dove MaxYear precede YearFounded, seq()
# conta ALL'INDIETRO e genera anni-azienda precedenti alla fondazione: 245
# righe su 106 aziende, di cui 186 arrivavano fino al panel finale. Sono dati
# sporchi - 88 di quelle aziende hanno YearFounded nel 2024 o 2025 con dati di
# anni precedenti, tipicamente ri-registrazioni.
# Alzando il limite all'anno di fondazione, il panel di un'azienda non puo'
# finire prima di iniziare: quelle 106 restano con una riga sola, l'anno zero,
# invece di sparire.
limite = pl.max_horizontal("MaxYear", "YearFounded")

scheletro = (
    vita
    # r_seq restituisce una LISTA di anni per ogni riga...
    .with_columns(r_seq(pl.col("YearFounded"), limite).alias("Year_Delta"))
    # ...ed explode la apre in una riga per anno. empty_as_null tiene l'azienda
    # anche se la lista e' vuota, con un anno nullo.
    .explode("Year_Delta", empty_as_null=True)
    # Delta = eta' dell'azienda in quell'anno. Alla fase 5 diventera' Age.
    .with_columns((pl.col("Year_Delta") - pl.col("YearFounded")).alias("Delta"))
    .select("CompanyID", "YearFounded", "Year_Delta", "Delta")
)
print(f"scheletro: {scheletro.height:,} righe azienda-anno")
print(f"righe con Delta < 0 (bug B6): {scheletro.filter(pl.col('Delta') < 0).height}")

In [ ]:
# Dopo il blocco 1.4 lo scheletro è una griglia nuda: CompanyID, YearFounded, Year_Delta (anno di calendario), Delta (età). Sa chi esisteva e quando, ma niente su cosa gli succedeva.

# Company.csv ha una colonna OwnershipStatus — «Privately Held», «Acquired/Merged», «Out of Business» — che però non ha una dimensione temporale: è una riga per azienda, lo stato attuale. 
# Insieme c'è OwnershipStatusDate, cioè quando quello stato è stato raggiunto.

# ── 1.5 · l'unico join per anno che sopravvive ────────────────────────────
# Il notebook completo fa cinque join per anno: stato finanziario, stato di
# business, proprieta', ultima valutazione e bilanci. Qui ne serve UNO solo -
# la proprieta' - perche' e' l'unico che finisce (indirettamente) nelle 53:
# alimenta i primi tre rami della cascata GrowthStage.


anno_proprieta = (
    azienda
    .select(
        "CompanyID",
        pl.col("OwnershipStatusDate").dt.year().alias("_anno_own"),
        "OwnershipStatus",
    )
    # Una chiave di join nulla non deve agganciare niente. In polars i null non
    # si abbinano fra loro di default, quindi e' una sicurezza, non una necessita'.
    .drop_nulls("_anno_own")
)

# Left join: lo stato di proprieta' compare SOLO sulla riga dell'anno della sua
# data, non su tutti gli anni dell'azienda. Su tutte le altre righe resta nullo.
scheletro = scheletro.join(
    anno_proprieta,
    left_on=["CompanyID", "Year_Delta"],
    right_on=["CompanyID", "_anno_own"],
    how="left",
)
print(f"righe con OwnershipStatus valorizzato: {scheletro['OwnershipStatus'].is_not_null().sum():,}"
      f" su {scheletro.height:,}")

In [ ]:
# ── 1.6 · il filtro sull'anno di fondazione, e la scrittura ───────────────
# db_master_1 ridotto a sei colonne: due feature e quattro che servono a valle.

# Mantiene solo le aziende fondate nell'anno minimo o dopo, e le scrive in un parquet che verra' usato in 2b e 4. Lo stesso filtro viene applicato allo scheletro,
#  perche' le aziende fondate prima non devono entrare nel panel finale.

db_master_1 = azienda.select(
    "CompanyID", "YearFounded", "HQCountry", "PrimaryIndustrySector",
    "OwnershipStatus", "OwnershipStatusDate",
).filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

# Lo stesso filtro sullo scheletro, con la stessa costante. Nell'R queste due
# righe usavano > 1999 e le fasi 2b e 4 usavano > 2000: era il difetto B1.
scheletro = scheletro.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

db1.write_parquet(cfg.interim("db1.parquet"))
db_master_1.write_parquet(cfg.interim("db_master_1.parquet"))
scheletro.write_parquet(cfg.interim("scheletro.parquet"))

print(f"db_master_1: {db_master_1.height:,} righe x {db_master_1.width} colonne  (attese 116.920)")
print(f"scheletro  : {scheletro.height:,} righe x {scheletro.width} colonne")

del azienda, vita, anno_proprieta, db1, db_master_1
gc.collect()

---
## Fase 2a — la tabella persona-azienda (`db3`)

Una riga per **coppia (azienda, persona)**, con gli attributi di quella persona
e la finestra di anni in cui era presente.

`db3` passa da 54 colonne a 22: spariscono biografia, indirizzo, ateneo di
`Person.csv`, `RolesCount_Total`, `Is_Other` (sempre falsa) e i contatori
grezzi, che vengono consumati subito dall'indice di esperienza.

In [ ]:
# ── 2a.1 · gli incarichi diventano una riga per coppia ────────────────────
# CompanyBoardTeamRelation ha una riga per INCARICO: la stessa persona nella
# stessa azienda compare piu' volte se ha cambiato ruolo (535.568 righe ->
# 534.851 coppie distinte). Qui si fondono.
COLONNE_BOARD = [
    "CompanyID", "PersonID",     # la coppia, che e' anche la chiave della deduplica
    "PersonName",                # serve agli override Ph.D / JD / MD del blocco 2a.5
    "FullTitle",                 # serve a IsFounder (2a.6)
    "IsCurrent",                 # serve all'imputazione di EndDate (2a.9)
    "StartDate", "EndDate",      # la finestra di presenza
]
COPPIA = ["CompanyID", "PersonID"]

# Nota il token set: qui l'R usa R_NA, SENZA "NaN". Un dettaglio che cambia
# quali celle diventano nulle, quindi va rispettato.
board = as_na(read_raw(cfg, "CompanyBoardTeamRelation", COLONNE_BOARD), R_NA)
print(f"righe grezze: {board.height:,}   coppie distinte: {board.select(COPPIA).n_unique():,}")

# is_duplicated() marca TUTTE le righe di una coppia che compare piu' di una volta.
duplicate = board.filter(board.select(COPPIA).is_duplicated())

if cfg.fix_dup_coalesce:
    # Corretto: si riprendono solo le righe delle coppie davvero duplicate.
    da_fondere = board.join(duplicate.select(COPPIA).unique(), on=COPPIA, how="semi")
else:
    # Bug B9, prima anomalia: l'R filtra per PersonID DA SOLO, non per la coppia.
    # Cosi' tira dentro anche le righe della stessa persona in ALTRE aziende
    # (2.175 righe invece di 1.430). Sul risultato non cambia niente - formano
    # gruppi da una riga sola - cambia solo l'ordine delle righe in uscita.
    da_fondere = board.filter(pl.col("PersonID").is_in(duplicate["PersonID"].implode()))

ALTRE = [c for c in COLONNE_BOARD if c not in COPPIA]
fuse = (
    da_fondere
    # arrange(PersonID, CompanyID) dell'R. L'ordinamento decide chi e' "first"
    # e chi "last" dentro ogni gruppo, quindi non e' cosmetico.
    .sort(["PersonID", "CompanyID"], maintain_order=True)
    .group_by(COPPIA, maintain_order=True)
    # Bug B9, seconda anomalia: coalesce(first(x), last(x)) e' POSIZIONALE, non
    # e' "il primo valore presente". Legge la prima riga del gruppo e, se e'
    # vuota, l'ultima: con tre o piu' incarichi la riga di mezzo e' invisibile.
    # Con due righe (709 coppie su 713) e' invece corretto.
    .agg([coalesce_first_last(c).alias(c) for c in ALTRE])
    .select(COLONNE_BOARD)          # rimette le colonne nell'ordine originale
)

# rbind(fuse, board) + distinct(keep = first): le righe fuse stanno davanti,
# quindi vincono; delle coppie non duplicate sopravvive l'originale.
board = pl.concat([fuse, board]).unique(subset=COPPIA, keep="first", maintain_order=True)
print(f"dopo la deduplica: {board.height:,} righe   (attese 534.851)")
del duplicate, da_fondere, fuse

In [ ]:
# ── 2a.2 · le date della permanenza ───────────────────────────────────────
db3 = (
    board
    .with_columns(parse_date_r(pl.col(c)).alias(c) for c in ("StartDate", "EndDate"))
    # Questo ordinamento conta anche a valle: decide in che ordine gli atenei
    # finiranno concatenati in Institute alla fase 2b. nulls_last riproduce il
    # comportamento di arrange() dell'R, che manda gli NA in coda.
    .sort(["CompanyID", "StartDate"], nulls_last=True)
)
del board
gc.collect()
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

In [ ]:
# ── 2a.3 · gli attributi della persona e l'indice di esperienza ───────────
# Gli otto contatori non sono fra le 53, ma sono gli INGREDIENTI di
# WorkExperienceIndex, che lo e' (via WorkExp_Idx_Mean e WorkExperienceIndex_CEO).
# Sono conteggi alla data di estrazione: e' il look-ahead strutturale (voce M0).
CONTATORI = [
    "CurrentPositionsCount", "FormerPositionsCount",
    "CurrentBoardSeatsCount", "FormerBoardSeatsCount",
    "CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
    "AffiliatedDealsCount", "NumberOfAffiliatedFunds",
]

persona = as_na(
    read_raw(cfg, "Person", ["PersonID", "Gender", *CONTATORI]), R_NA_NAN
).with_columns(to_num(c) for c in CONTATORI)      # da stringa a Float64

db3 = db3.join(persona, on="PersonID", how="left")
del persona
gc.collect()
print(f"persone assenti da Person.csv: {db3['Gender'].is_null().sum():,} righe")


def somma_righe(colonne: list[str]) -> pl.Expr:
    """rowSums(..., na.rm = TRUE): una riga tutta mancante fa 0, non nullo.

    Conseguenza da tenere a mente: una persona che non esiste in Person.csv
    ottiene zero ruoli, indistinguibile da una che davvero non ne ha avuti."""
    return pl.sum_horizontal([pl.col(c).fill_null(0.0) for c in colonne])


db3 = db3.with_columns(
    somma_righe(["CurrentPositionsCount", "FormerPositionsCount"]).alias("_positions"),
    somma_righe(["CurrentBoardSeatsCount", "FormerBoardSeatsCount"]).alias("_boardseats"),
    somma_righe(["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
                 "AffiliatedDealsCount", "NumberOfAffiliatedFunds"]).alias("_otherroles"),
).with_columns(
    # scale_r = scale() di R: media zero, e deviazione standard CAMPIONARIA
    # (denominatore n-1, mentre numpy e polars usano n). La standardizzazione e'
    # GLOBALE su tutto db3, non per azienda: rifarla per gruppo darebbe numeri
    # completamente diversi. Il log(x+1) comprime code lunghissime.
    scale_r((pl.col("_positions") + 1).log()).alias("_pos_z"),
    scale_r((pl.col("_boardseats") + 1).log()).alias("_brd_z"),
    scale_r((pl.col("_otherroles") + 1).log()).alias("_oth_z"),
).with_columns(
    # rowMeans(..., na.rm = TRUE) sui tre z-score. Secondo with_columns perche'
    # le tre colonne _*_z devono esistere prima di poterne fare la media.
    pl.mean_horizontal("_pos_z", "_brd_z", "_oth_z").alias("WorkExperienceIndex")
).drop(
    # I contatori e le intermedie hanno finito il loro lavoro: l'indice e' calcolato.
    # Nel panel completo restavano dentro db3 senza che nessuno le leggesse.
    *CONTATORI, "_positions", "_boardseats", "_otherroles", "_pos_z", "_brd_z", "_oth_z",
)
print(f"WorkExperienceIndex: media {db3['WorkExperienceIndex'].mean():.4f}")

In [ ]:
# ── 2a.4 · il titolo di studio e il campo di studi ────────────────────────
# Due catene di grepl a CORTO CIRCUITO: il primo che matcha vince e gli altri
# non vengono nemmeno valutati. L'ordine delle regole e' quindi parte della
# logica: "MD" compare nella prima regola, quindi un MD non arriva mai a
# "Master's" anche se la seconda regola contiene "Master".
REGOLE_TITOLO = [
    (r"PhD|Doctor|MD|PsyD|DPhil|DC|DDS|DPT|OD|JD", "PhD/Doctorate"),
    (r"MBA|LLM|Master|MSc|MPhil|MPA|MFA|MEng|MAcc|Graduate", "Master's"),
    (r"Bachelor|BSc|BEng|Laurea|BS|BFA|BCom|degree|Business Program|Undergrad", "Bachelor's"),
    (
        r"Certified|Certificat|A Levels|A-Levels|Diplom|DEA|DESS|Dipl\.-Ing|"
        r"Executive Development Program|Executive Education|Executive Education program|"
        r"Executive Program|First Legal State Exam|Legal Practice Course|Vordiplom",
        "Diploma/Certificate",
    ),
]
# La gerarchia: Highest_Degree e' l'indice 1-based del livello piu' alto raggiunto.
# Il ramo finale "Other" cattura anche un titolo mancante, quindi DegreeLevel non
# e' mai nullo: chi non dichiara nulla prende 1, non NA.
GERARCHIA = ["Other", "Diploma/Certificate", "Bachelor's", "Master's", "PhD/Doctorate"]

REGOLE_AREA = [
    (r"business|management|bank|invest|financ|marketing|real estate|account|"
     r"Entrepreneur|commerce|econom|actuarial science|private equity", "Economics"),
    (r"engineer|civil|electric|mechanic|electronic|Operations Research|material|"
     r"logistic|Engeneering", "Engineering"),
    (r"statistic|machine learning|Natural Language|robot|technolog|comput|data|"
     r"informatic|Artificial Intelligence|information science|information systems|"
     r"data science|softwar", "IT and Computer Science"),
    (r"law|tax|justice|forensic|legal|jurisprudence|intellectual property", "Law"),
    (r"medic|nursing|pharmac|health|immunolog|neuroscien|genetic|physio", "Health and Medicine"),
    (r"social science|strateg|sociology|psychology|anthropology|international relations|"
     r"polit|government|geography|polic|international|foreign service|social studies|"
     r"criminology|cognitive science|public affairs|urban planning|social work|"
     r"human resource|leadership|foreign", "Social Sciences"),
    (r"natural|biolog|chemistry|physic|environmental science|math|geology|life science|"
     r"zoology|agriculture", "Natural Sciences"),
    (r"humanit|literature|histor|philosoph|language|linguist|english|spanis|american|"
     r"religion|classic|french|theolog|cultural studies|europe|arts|design|music|"
     r"architecture|journalism|media|public relations|advertising|communic|education|"
     r"early childhood|special education|administration", "Humanities and Arts"),
]

studi = as_na(
    read_raw(cfg, "PersonEducationRelation",
             ["PersonID", "Degree", "Major_Concentration", "GraduatingYear", "Institute"]),
    R_NA,
)
# r_case_when applica le regole in ordine; chi non matcha niente finisce in "Other".
studi = studi.with_columns(
    r_case_when(REGOLE_TITOLO, pl.col("Degree"), pl.lit("Other")).alias("DegreeLevel")
)

titolo, area = pl.col("Degree"), pl.col("Major_Concentration")
studi = studi.with_columns(
    # Se l'area di studi manca, l'R prova a dedurla dal nome del titolo.
    # fill_null(False) perche' str.contains su un valore nullo da' null, e
    # pl.when tratterebbe quel null come "non matcha" - qui e' quello che vogliamo,
    # ma scriverlo esplicito evita di doverci ripensare.
    pl.when(area.is_null() & titolo.str.contains("(?i)law").fill_null(False)).then(pl.lit("Law"))
    .when(area.is_null() & titolo.str.contains("(?i)MBA").fill_null(False)).then(pl.lit("Business"))
    .when(area.is_null() & titolo.str.contains("(?i)Medicine").fill_null(False)).then(pl.lit("Medicine"))
    .otherwise(area)
    .alias("Major_Concentration")
)
studi = studi.with_columns(
    # Primo ramo del case_when dell'R: se l'area e' ancora nulla, Field e' nulla.
    # Non "Other": nulla. La differenza conta, perche' i flag Is_* qui sotto
    # distinguono "nessun titolo classificabile" da "titolo di area Other".
    pl.when(pl.col("Major_Concentration").is_null())
    .then(None)
    .otherwise(r_case_when(REGOLE_AREA, pl.col("Major_Concentration"), pl.lit("Other")))
    .alias("Field")
)
print(studi["Field"].value_counts(sort=True))

In [ ]:
# ── 2a.5 · l'istruzione, aggregata per persona ────────────────────────────
# Una persona ha in media 1,7 titoli: qui si collassano in una riga.
# Nota: Is_Other del panel completo NON e' qui. Cercava il valore "Other/Unknown"
# in Field, che Field non produce mai (produce "Other"): era sempre falsa e non
# e' fra le 53. Eliminata insieme al suo bug (B2).
AREE = {
    "Is_Eco": "Economics",
    "Is_Eng": "Engineering",
    "Is_Med": "Health and Medicine",
    "Is_Hum": "Humanities and Arts",
    "Is_IT": "IT and Computer Science",
    "Is_Law": "Law",
    "Is_NS": "Natural Sciences",
    "Is_SS": "Social Sciences",
}

# Da "Master's" all'indice 4. default=None perche' un valore fuori gerarchia
# sarebbe un errore, non uno zero.
indice_titolo = (
    pl.col("DegreeLevel")
    .replace_strict({nome: i + 1 for i, nome in enumerate(GERARCHIA)}, default=None)
    .cast(pl.Int64)
)
# La guardia dell'R: ifelse(sum(!is.na(Field)) > 0, any(...), NA). Se la persona
# non ha nessun campo di studi classificabile i flag sono NULLI, non falsi.
ha_campo = pl.col("Field").is_not_null().sum() > 0

istruzione = studi.group_by("PersonID").agg(
    [
        pl.when(ha_campo).then(pl.col("Field").eq(valore).any()).otherwise(None).alias(flag)
        for flag, valore in AREE.items()
    ]
    + [
        # Earliest_Year: il primo anno di laurea. Guarda indietro, non avanti:
        # e' l'unico attributo della persona che NON e' look-ahead.
        pl.col("GraduatingYear").cast(pl.Float64, strict=False).min().alias("Earliest_Year"),
        indice_titolo.max().alias("Highest_Degree"),
        # paste(na.omit(Institute), collapse = "; "): QUI i mancanti si scartano.
        # Alla fase 2b lo stesso paste li includera' come testo "NA" (bug B7).
        # Due comportamenti opposti a due fasi di distanza.
        pl.when(pl.col("Institute").is_not_null().sum() > 0)
        .then(pl.col("Institute").drop_nulls().str.join("; "))
        .otherwise(None)
        .alias("Institute"),
    ]
)
db3 = db3.join(istruzione, on="PersonID", how="left")
del studi, istruzione
gc.collect()
print(db3["Highest_Degree"].value_counts(sort=True).sort("Highest_Degree"))

In [ ]:
# ── 2a.6 · gli override dal nome della persona ────────────────────────────
# Se il nome contiene un titolo, l'R lo impone a 5 (PhD/Doctorate) scavalcando
# la tabella dell'istruzione.
nome = pl.col("PersonName")
ha_phd = nome.str.contains(r"Ph\.?D").fill_null(False)
# literal=True: " JD" e " MD" sono cercati alla lettera e CASE-SENSITIVE, a
# differenza di tutti gli altri grepl del blocco che hanno ignore.case = TRUE.
# Non e' una svista da correggere: e' il comportamento da riprodurre.
ha_jd = nome.str.contains(" JD", literal=True).fill_null(False)
ha_md = nome.str.contains(" MD", literal=True).fill_null(False)

db3 = db3.with_columns(
    # L'R fa tre assegnazioni in sequenza; l'OR le condensa con lo stesso esito.
    pl.when(ha_phd | ha_jd | ha_md).then(5).otherwise(pl.col("Highest_Degree")).alias("Highest_Degree"),
    pl.when(ha_jd).then(True).otherwise(pl.col("Is_Law")).alias("Is_Law"),
    pl.when(ha_md).then(True).otherwise(pl.col("Is_Med")).alias("Is_Med"),
)
print(f"nomi con Ph.D: {db3.select(ha_phd).to_series().sum():,}  "
      f"' JD': {db3.select(ha_jd).to_series().sum():,}  "
      f"' MD': {db3.select(ha_md).to_series().sum():,}")

In [ ]:
# ── 2a.7 · IsFounder ──────────────────────────────────────────────────────
posizioni = as_na(
    read_raw(cfg, "PersonPositionRelation", ["PersonID", "EntityID", "PositionLevel"]),
    R_NA_NAN,
# distinct(EntityID, PersonID): una persona puo' avere piu' posizioni nella
# stessa entita', qui ne serve una sola.
).unique(subset=["EntityID", "PersonID"], keep="first", maintain_order=True)

# Il join e' sulla COPPIA: la posizione di quella persona in QUELLA azienda.
db3 = db3.join(
    posizioni, left_on=["CompanyID", "PersonID"], right_on=["EntityID", "PersonID"], how="left"
)
del posizioni
gc.collect()

# paste(FullTitle, PositionLevel, sep = "; ") in R stringifica i mancanti come
# il testo "NA", quindi la stringa risultante non e' MAI nulla e IsFounder non
# e' mai nullo: e' False, non NA, per chi non ha ne' titolo ne' posizione.
qualifica = pl.concat_str(
    [pl.col("FullTitle").fill_null("NA"), pl.col("PositionLevel").fill_null("NA")], separator="; "
)
db3 = db3.with_columns(qualifica.str.contains("(?i)Found").alias("IsFounder"))
# PersonName, FullTitle e PositionLevel hanno finito: servivano solo qui e in 2a.6.
db3 = db3.drop("PersonName", "FullTitle", "PositionLevel")
print(f"founder: {db3['IsFounder'].sum():,}   IsFounder nullo: {db3['IsFounder'].null_count()}")

In [ ]:
# ── 2a.8 · le aziende fallite ─────────────────────────────────────────────
# Serve a imputare EndDate: chi lavorava in un'azienda fallita ha smesso quando
# l'azienda e' fallita.
aziende = pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
    "CompanyID", "OwnershipStatus", "OwnershipStatusDate"
)
fallite = aziende.filter(pl.col("OwnershipStatus") == "Out of Business").with_columns(
    pl.lit(True).alias("Is_Out"),
    pl.col("OwnershipStatusDate").alias("OutDate"),
)
# ATTENZIONE: db_master_1 e' la versione FILTRATA per anno di fondazione.
# Costruire questa tabella dall'anagrafica completa accenderebbe Is_Out su piu'
# aziende e cambierebbe l'imputazione di EndDate.
print(f"aziende fuori mercato: {fallite.height:,}")

# Bug B10: e' un LEFT join, quindi per tutte le altre aziende Is_Out resta
# NULLO, non False. Al blocco 2a.9 la condizione "Is_Out == FALSE" varra' null
# e neutralizzera' una delle regole. Un catch-all successivo recupera i casi,
# quindi il bug si auto-sana - ma solo per caso.
db3 = db3.join(fallite.select("CompanyID", "Is_Out", "OutDate"), on="CompanyID", how="left")
del aziende, fallite
print(f"righe con Is_Out nullo: {db3['Is_Out'].null_count():,} su {db3.height:,}")

In [ ]:
# ── 2a.9 · YearFounded, e l'imputazione di StartDate ──────────────────────
# YearFounded arriva da db1, la tabella NON filtrata (vedi 1.3).
db3 = db3.join(pl.read_parquet(cfg.interim("db1.parquet")), on="CompanyID", how="left")

inizio_fondazione = pl.date(pl.col("YearFounded"), 1, 1)
inizia_prima = pl.col("StartDate").dt.year() < pl.col("YearFounded")

# La trappola peggiore di tutta la pipeline.
# r_if_else, non pl.when: if_else() di dplyr restituisce NA quando la
# CONDIZIONE e' NA. Per un'azienda senza YearFounded, "StartDate < NA" vale NA,
# l'intero OR vale NA, e l'R CANCELLA una StartDate perfettamente valida.
# pl.when tratterebbe la condizione nulla come falsa e terrebbe il valore:
# 5.357 righe di differenza.
db3 = db3.with_columns(
    r_if_else(
        pl.col("StartDate").is_null() | inizia_prima,
        inizio_fondazione,
        pl.col("StartDate"),
    ).alias("StartDate")
)
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

In [ ]:
# ── 2a.10 · l'imputazione di EndDate, quattro regole in cascata ───────────
fine_2024 = pl.date(2024, 12, 31)

db3 = db3.with_columns(
    # Regola 1: chi risulta ancora in carica in un'azienda viva arriva a fine 2024.
    # Quella data dipende dal VINTAGE dell'estrazione: rieseguire su un download
    # del 2026 darebbe finestre diverse per le stesse persone (voce M4).
    # ~pl.col("Is_Out") e' NULLO dove Is_Out e' nullo (bug B10), e in polars
    # come in R  True & True & null = null: la regola non scatta.
    r_if_else(
        pl.col("EndDate").is_null() & (pl.col("IsCurrent") == "Yes") & ~pl.col("Is_Out"),
        fine_2024,
        pl.col("EndDate"),
    ).alias("EndDate")
).with_columns(
    # Regola 2: azienda fallita -> si esce quando fallisce.
    r_if_else(
        pl.col("EndDate").is_null() & pl.col("Is_Out"),
        pl.col("OutDate"),
        pl.col("EndDate"),
    ).alias("EndDate")
)

durata = pl.col("EndDate").dt.year() - pl.col("StartDate").dt.year()

if cfg.fix_permanenza_media_per_company:
    permanenza = durata.mean().round(0).over("CompanyID")
else:
    # Bug B5: l'R usa summarise() SENZA group_by, nonostante il commento dica
    # "per ciascuna CompanyID". Il risultato e' UN SOLO NUMERO per tutto il
    # dataset - vale 7 anni, media di 6,606 sulle 124.277 righe che a questo
    # punto hanno entrambe le date - usato per imputare la fine di chiunque,
    # che sia una startup morta in due anni o una sopravvissuta quindici.
    # La regola che lo usa scatta su 115.399 righe, il 21,6% di db3.
    permanenza = pl.lit(
        db3.filter(pl.col("EndDate").is_not_null() & pl.col("StartDate").is_not_null())
        .select(durata.mean().round(0))
        .item()
    )

db3 = db3.with_columns(permanenza.alias("_permanenza"))
print(f"permanenza media usata per l'imputazione: {db3['_permanenza'][0]} anni")

# pmin(inizio + permanenza, 2024-12-31): l'imputazione non puo' superare il
# vintage dell'estrazione.
fine_imputata = pl.min_horizontal(
    pl.date(pl.col("StartDate").dt.year() + pl.col("_permanenza").cast(pl.Int64), 1, 1), fine_2024
)
db3 = db3.with_columns(
    # Regola 3: chi non e' piu' in carica esce dopo la permanenza media.
    pl.when(
        pl.col("EndDate").is_null()
        & (pl.col("IsCurrent") == "No")
        & pl.col("StartDate").is_not_null()
        & pl.col("_permanenza").is_not_null()
    )
    .then(fine_imputata)
    # Regola 4, il catch-all: tutto il resto arriva a fine 2024. E' questo che
    # auto-sana il bug B10 - le righe che la regola 1 non ha toccato per via del
    # null finiscono comunque qui, con lo stesso esito.
    .when(pl.col("EndDate").is_null() & pl.col("StartDate").is_not_null())
    .then(fine_2024)
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).drop("_permanenza", "Is_Out", "OutDate", "IsCurrent")   # hanno finito il loro lavoro
print(f"EndDate valorizzate: {db3['EndDate'].is_not_null().sum():,} su {db3.height:,}")

In [ ]:
# ── 2a.11 · la finestra in anni di vita dell'azienda ──────────────────────
db3 = db3.with_columns(
    # Raddrizza le finestre impossibili. Ne nascono dalla deduplica di 2a.1,
    # che prende StartDate da una riga ed EndDate da un'altra: 7 coppie su 713.
    pl.when(pl.col("EndDate") < pl.col("StartDate"))
    .then(pl.col("StartDate"))
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).with_columns(
    # Da date di calendario a ETA' DELL'AZIENDA: e' l'unita' di misura del panel.
    (pl.col("StartDate").dt.year() - pl.col("YearFounded")).alias("DeltaStart"),
    (pl.col("EndDate").dt.year() - pl.col("YearFounded")).alias("DeltaEnd"),
)

db3 = db3.with_columns(
    # Si assume che un founder ci sia dal primo giorno. Sovrascrive 19.458 date
    # di inizio reali e diverse da zero (voce M5).
    pl.when(pl.col("IsFounder")).then(0).otherwise(pl.col("DeltaStart")).alias("DeltaStart")
).drop("StartDate", "EndDate")     # da qui in poi contano solo i due Delta

db3.write_parquet(cfg.interim("db3.parquet"))
print(f"db3: {db3.height:,} righe x {db3.width} colonne   (il panel completo ne ha 54)")
print(f"colonne: {db3.columns}")

---
## Fase 2b — le colonne di team, anno per anno

Ogni persona viene **espansa** su tutti gli anni della sua finestra, poi si
collassa per `(azienda, anno)`.

Dei 23 aggregati del panel completo ne restano **13**: spariscono
`Highest_Degree_Max`, `RolesCount_Max`, `RolesCount_Mean`, `WorkExp_Idx_Max`,
`Positions`, `BoardSeats`, `OtherRoles` (medie di team) e `Is_Other`.

In [ ]:
# ── 2b.1 · l'espansione a (azienda, anno, persona) ────────────────────────
db3 = pl.read_parquet(cfg.interim("db3.parquet"))

# expand_team costruisce due griglie e le unisce:
#   - per ogni AZIENDA, ogni anno da min(DeltaStart) a max(DeltaEnd) del suo team
#   - per ogni PERSONA, ogni anno della sua finestra
# left join su (azienda, anno) -> una riga per persona presente quell'anno.
# Il filtro sulla soglia sta dentro la funzione perche' e' cio' che tiene il
# risultato dentro la memoria disponibile. Ha anche un secondo effetto: su un
# anno-azienda che non ha trovato NESSUNA persona, YearFounded arriva dal lato
# persona ed e' nullo, e un confronto "> soglia" scarta i null - e' cosi' che
# non nascono righe fantasma con Total_People = 1 e tutto il resto vuoto.
# expand_team applica la soglia con un confronto STRETTO ("> soglia"), perche'
# e' condivisa con build_panel.ipynb che e' congelato e non si tocca: le si
# passa quindi l'anno precedente, che e' lo stesso insieme di aziende.
espanso = expand_team(db3, founding_year_threshold=ANNO_MIN_FONDAZIONE - 1)
del db3
gc.collect()
print(f"righe (azienda, anno, persona): {espanso.height:,}")

In [ ]:
# ── 2b.2 · i tredici aggregati di team ────────────────────────────────────
FLAG_AREE = ["Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT"]

istituto = pl.col("Institute")
if not cfg.fix_institute_na_literal:
    # Bug B7: qui il paste dell'R NON scarta i mancanti, li stringifica come il
    # testo "NA". Da cui i 390.544 Institute che cominciano con "NA; ".
    # Cosmetico: a valle Institute viene spezzato su ";" e cercato per
    # appartenenza fra le top 50, e "NA" non e' il nome di nessuna universita'.
    istituto = istituto.fill_null("NA")

totale = pl.len()
team = espanso.group_by(["CompanyID", "Years"]).agg(
    # .N dell'R: quante persone risultano presenti in quell'anno-azienda.
    totale.alias("Total_People"),
    # Il denominatore e' TOTALE, non "le persone di genere noto": se il genere
    # e' ignoto la quota di donne e' diluita verso il basso. Misurato: il genere
    # manca solo sullo 0,6% delle righe, quindi in pratica non sposta niente.
    (pl.col("Gender").eq("Female").sum() / totale * 100).alias("Percent_Females"),
    # any(x, na.rm = TRUE): un gruppo tutto mancante da' False, non nullo.
    # fill_null(False) prima di any() e' esattamente questo.
    *[pl.col(c).fill_null(False).any().alias(c) for c in FLAG_AREE],
    # In R max()/mean() su un gruppo tutto-NA danno -Inf e NaN, che un ciclo
    # successivo riconverte in NA. In polars danno gia' null: stesso esito.
    pl.col("Earliest_Year").mean().alias("Avg_Earliest_Year"),
    pl.col("Highest_Degree").mean().alias("Highest_Degree_Mean"),
    # paste(unique(Institute), collapse = "; "): l'ordine dipende dall'ordine
    # delle righe, che viene dal sort del blocco 2a.2. Non porta informazione -
    # a valle si fa split(";") e poi un set - ma va tenuto stabile.
    istituto.unique(maintain_order=True).str.join("; ").alias("Institute"),
    pl.col("WorkExperienceIndex").mean().alias("WorkExp_Idx_Mean"),
    # sum() su booleani conta i True e ignora i null: e' sum(x, na.rm = TRUE).
    pl.col("IsFounder").sum().alias("Total_Founders"),
)
del espanso
gc.collect()

# Quando nessuna persona del gruppo ha un ateneo, il join di stringhe produce ""
# invece che null. L'R lo risolve col ciclo dei token mancanti; qui basta questo.
team = team.with_columns(
    pl.when(pl.col("Institute") == "").then(None).otherwise(pl.col("Institute")).alias("Institute")
)
print(f"team: {team.height:,} anni-azienda x {team.width} colonne")

In [ ]:
# ── 2b.3 · il full join con lo scheletro ──────────────────────────────────
scheletro = pl.read_parquet(cfg.interim("scheletro.parquet"))
print(f"scheletro: {scheletro.height:,} righe   team: {team.height:,} righe")

panel = (
    scheletro.join(
        team.rename({"Years": "Delta"}),
        on=["CompanyID", "Delta"],
        # FULL, non left: il team puo' avere anni-azienda che lo scheletro non
        # ha (una persona presente prima di MaxYear o dopo). E' cosi' che si
        # arriva a 1.001.625 righe; con un left join sarebbero meno.
        how="full",
        coalesce=True,           # una sola colonna CompanyID/Delta, non due
    )
    .sort(["CompanyID", "Delta"])
    # Le righe che arrivano dal solo lato team non hanno YearFounded ne'
    # Year_Delta. YearFounded e' costante per azienda, quindi si propaga in
    # avanti e poi all'indietro (fill .direction = "downup" dell'R)...
    .with_columns(pl.col("YearFounded").fill_null(strategy="forward").over("CompanyID"))
    .with_columns(pl.col("YearFounded").fill_null(strategy="backward").over("CompanyID"))
    # ...e l'anno di calendario si ricostruisce dall'eta'.
    .with_columns(pl.col("Year_Delta").fill_null(pl.col("YearFounded") + pl.col("Delta")))
    .sort(["CompanyID", "Delta"])
)
del team, scheletro
gc.collect()

panel.write_parquet(cfg.interim("panel_team.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne   (attese 1.001.625)")
print(f"righe con dati di team: {panel['Total_People'].is_not_null().sum():,}")
del panel
gc.collect()

---
## Fase 4 — deal e investitori

I round di finanziamento. Sono loro a determinare `GrowthStage`, cioè il target.

**L'imputazione RandomForest non c'è**, ed è una decisione, non una mancanza: il
modello dell'R gira senza `set.seed`, è addestrato su deal di tutti gli anni,
è fittato prima di qualsiasi split, e ha il tipo di deal fra i predittori
mentre il tipo di deal determina il target. Cadono con lei `TotalRaised_Est` e
le altre sei `*_Est`, `UndisclosedAmountFlag` e la lettura di `DealSynopsis`.

Al posto di `TotalRaised_Est`, a valle, si userà **`TotalRaised`**.

In [ ]:
# ── 4.1 · gli investitori, in sette categorie ─────────────────────────────
# La mappa dell'R. Tutto cio' che non e' in mappa - compreso un tipo mancante -
# finisce in "Other", che non genera nessun flag.
CATEGORIA_INVESTITORE = {
    "Venture Capital": "Venture Capital",
    "Corporate Venture Capital": "Venture Capital",
    "Growth/Expansion": "Venture Capital",
    "Not-For-Profit Venture Capital": "Venture Capital",
    "VC-Backed Company": "Venture Capital",
    "Angel (individual)": "Angel",
    "Angel Group": "Angel",
    "Accelerator/Incubator": "Accelerator",
    "Corporation": "Corporate",
    "Corporate Development": "Corporate",
    "PE/Buyout": "Private Equity",
    "Family Office": "Private Equity",
    "PE-Backed Company": "Private Equity",
    "Holding Company": "Private Equity",
    "Merchant Banking Firm": "Private Equity",
    "Mezzanine": "Private Equity",
    "Secondary Buyer": "Private Equity",
    "Other Private Equity": "Private Equity",
    "Special Purpose Acquisition Company (SPAC)": "Private Equity",
    "Fundless Sponsor": "Private Equity",
    "Government": "Public Investor",
    "University": "Public Investor",
    "Sovereign Wealth Fund": "Public Investor",
    "Mutual Fund": "Public Investor",
}
# I sei flag has_*: il nome del flag e il valore di categoria che cerca.
# Angel e Accelerator non sono fra le 53 ma servono lo stesso: al blocco 4.7
# fanno un OR con Is_Angel e Is_Accelerator, che invece lo sono.
CATEGORIE = {
    "Angel": "Angel",
    "Corporate": "Corporate",
    "VentureCapital": "Venture Capital",
    "Accelerator": "Accelerator",
    "PrivateEquity": "Private Equity",
    "PublicInvestor": "Public Investor",
}
# Due sole grandezze numeriche: sono quelle che diventano MeanTotalInvestments_cum
# e MeanMedianRoundAmount_cum. Il panel completo ne legge quattro; le altre due
# (TotalActivePortfolio, MedianValuation) producono cumulate che nessuno usa.
NUMERICHE = ["TotalInvestments", "MedianRoundAmount"]

relazione = as_na(
    read_raw(cfg, "DealInvestorRelation",
             ["DealID", "InvestorID", "InvestorStatus", "IsLeadInvestor"]),
    R_NA_NAN,
)
investitori = as_na(
    read_raw(cfg, "Investor", ["InvestorID", "PrimaryInvestorType", *NUMERICHE]), R_NA_NAN
).with_columns(to_num(c) for c in NUMERICHE)

relazione = relazione.join(investitori, on="InvestorID", how="left").with_columns(
    pl.col("PrimaryInvestorType")
    .replace_strict(CATEGORIA_INVESTITORE, default="Other")
    .alias("InvestorCategory")
)
del investitori
print(f"partecipazioni a deal: {relazione.height:,}   deal distinti: {relazione['DealID'].n_unique():,}")

In [ ]:
# ── 4.2 · aggregare per deal ──────────────────────────────────────────────
nuovo = pl.col("InvestorStatus") == "New Investor"
lead = pl.col("IsLeadInvestor") == "Yes"

# Quasi tutti gli aggregati dell'R sono dentro un
#   ifelse(any(InvestorStatus == "New Investor"), <aggregato>, NA)
# e any() SENZA na.rm ha tre esiti, non due:
#   - TRUE  se almeno uno e' "New Investor"
#   - NA    se nessuno lo e' MA qualche valore manca
#   - FALSE altrimenti.
# Usato come condizione di un ifelse, l'NA rende NULLO l'aggregato intero.
condizione = (
    pl.when(nuovo.fill_null(False).any()).then(True)
    .when(nuovo.is_null().any()).then(None)
    .otherwise(False)
)


def media_sui_nuovi(colonna: str) -> pl.Expr:
    """La media calcolata sui soli nuovi investitori, nulla se la condizione e' NA."""
    return r_if_else(condizione, pl.col(colonna).filter(nuovo.fill_null(False)).mean(), None)


def ha_categoria(categoria: str, maschera: pl.Expr) -> pl.Expr:
    """C'e' almeno un investitore di quella categoria, fra quelli selezionati
    dalla maschera? Nullo se la condizione sui nuovi investitori e' NA."""
    appartiene = pl.col("InvestorCategory").filter(maschera.fill_null(False)) == CATEGORIE[categoria]
    return r_if_else(condizione, appartiene.fill_null(False).any(), None)


per_deal = relazione.group_by("DealID").agg(
    # Quanti investitori NUOVI in questo round. Diventera' il peso delle medie
    # ponderate cumulate della fase 5.
    nuovo.fill_null(False).sum().alias("TotalInvestors"),
    media_sui_nuovi("TotalInvestments").alias("MeanTotalInvestments"),
    media_sui_nuovi("MedianRoundAmount").alias("MeanMedianRoundAmount"),
    # I sei flag sui nuovi investitori.
    *[ha_categoria(c, nuovo).alias(f"has_{c}") for c in CATEGORIE],
    # I sei flag sui lead. Asimmetria dell'R, riprodotta: il FILTRO e' sui lead,
    # ma la CONDIZIONE che decide se restituire NA e' ancora quella sui nuovi
    # investitori. Non e' un refuso mio, e' cosi' nell'originale.
    *[ha_categoria(c, lead).alias(f"has_{c}_Lead") for c in CATEGORIE],
)
del relazione
gc.collect()
print(f"deal con almeno un investitore registrato: {per_deal.height:,}")

In [ ]:
# ── 4.3 · i deal ──────────────────────────────────────────────────────────
COLONNE_DEAL = [
    "CompanyID", "DealID",
    "DealNo",                  # il progressivo del round: serve alla riparazione delle date
    "DealDate",
    "DealType",                # da qui nascono tutti i flag e la regola Zero_Invested
    "TotalInvestedCapital",    # -> TotalRaised
    "CEOPBId",                 # -> CEO_ID, l'unico ponte fra i deal e le persone
]

deal = as_na(read_raw(cfg, "Deal", COLONNE_DEAL), R_NA_NAN).with_columns(
    pl.col("DealNo").cast(pl.Int64, strict=False),
    parse_date_r(pl.col("DealDate")).alias("DealDate"),
    to_num("TotalInvestedCapital"),
)
print(f"deal grezzi: {deal.height:,}   senza DealDate: {deal['DealDate'].is_null().sum():,}")

# Dall'anagrafica servono l'anno di fondazione (per il filtro e per collocare il
# deal) e lo stato di proprieta' con la sua data (per riparare le date mancanti).
deal = deal.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "YearFounded", "OwnershipStatus", "OwnershipStatusDate"
    ),
    on="CompanyID",
    how="left",
)

In [ ]:
# ── 4.4 · riparare le date dei deal: quattro passaggi ─────────────────────
# 61.169 deal su 385.481 non hanno una data. Questi quattro passaggi ne
# inventano una per 36.206 (il 10,9%); i restanti 16.690 usciranno dal panel.
FALLIMENTO = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business"]
ACQUISITA = ["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]
PRIMO_ROUND = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Capitalization",
               "Early Stage VC", "Seed Round", "Spin-Off"]

data_stato = pl.col("OwnershipStatusDate")
mancanti_iniziali = deal["DealDate"].is_null().sum()

# Passaggio 1: un deal di fallimento su un'azienda che risulta fallita prende la
# data del fallimento. Plausibile: 1.210 deal.
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(FALLIMENTO)
        & (pl.col("OwnershipStatus") == "Out of Business")
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)
# Passaggio 2: stessa idea per le acquisizioni. 234 deal.
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealType") == "Merger/Acquisition")
        & pl.col("OwnershipStatus").is_in(ACQUISITA)
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)

# Terzo e ultimo punto che filtra le aziende, con la stessa costante.
# Il filtro sta FRA il passaggio 2 e il 3, come nell'R: spostarlo cambierebbe
# quali deal ricevono una data inventata, quindi la posizione non si tocca.
deal = deal.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)
print(f"deal dopo il filtro YearFounded >= {ANNO_MIN_FONDAZIONE}: {deal.height:,}")

# Passaggio 3: il PRIMO round, se e' di tipo iniziale, viene messo al 1 gennaio
# dell'anno di fondazione. E' un'assunzione forte e tocca 23.147 deal: schiaccia
# quei round sull'eta' ZERO, che e' proprio la variabile con cui a valle si
# decide chi entra nel campione (StartingAge <= 2).
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(PRIMO_ROUND)
        & (pl.col("DealNo") == 1)
        & pl.col("YearFounded").is_not_null()
    ).then(pl.date(pl.col("YearFounded"), 1, 1)).otherwise(pl.col("DealDate")).alias("DealDate")
)

# Passaggio 4: la media arrotondata per eccesso fra l'anno del deal precedente e
# quello del successivo. Invenzione pura, 11.615 deal.
deal = deal.sort(["CompanyID", "DealNo"]).with_columns(pl.col("DealDate").dt.year().alias("_anno"))
# shift(1) guarda la riga precedente dello stesso gruppo, shift(-1) la successiva.
anno_prec = pl.col("_anno").shift(1).over("CompanyID")
anno_succ = pl.col("_anno").shift(-1).over("CompanyID")
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealNo") > 1)
        & anno_prec.is_not_null()
        & anno_succ.is_not_null()
    ).then(pl.date(((anno_prec + anno_succ) / 2).ceil().cast(pl.Int64), 1, 1))
    .otherwise(pl.col("DealDate")).alias("DealDate")
).drop("_anno", "OwnershipStatus", "OwnershipStatusDate")

print(f"date mancanti: {mancanti_iniziali:,} -> {deal['DealDate'].is_null().sum():,}")

In [ ]:
# ── 4.5 · collocare il deal in un anno del panel ──────────────────────────
anno_deal = pl.col("DealDate").dt.year()

deal = deal.with_columns(
    # pmax(year(DealDate), YearFounded) SENZA na.rm: se l'anno manca il
    # risultato e' NULLO, il deal finisce in un gruppo ad anno nullo e al join
    # con il panel non si aggancia a niente. Spariscono cosi', in silenzio,
    # 16.690 deal su 13.184 aziende - fra cui 1.912 Later Stage VC e 1.272
    # Early Stage VC, che avrebbero alzato lo stadio di quelle aziende.
    # pl.max_horizontal da solo ignora i null e restituirebbe l'anno di
    # fondazione, parcheggiando quei deal sull'anno zero: comportamento diverso,
    # e nemmeno quello giusto. Il when esplicito serve a non farlo.
    pl.when(anno_deal.is_null())
    .then(None)
    .otherwise(pl.max_horizontal(anno_deal, pl.col("YearFounded")))
    .alias("Year_Delta"),
)
# I deal datati PRIMA della fondazione vengono schiacciati sull'anno zero
# invece che scartati: 143 deal, trascurabile.
print(f"deal senza anno, che escono dal panel: {deal['Year_Delta'].is_null().sum():,}")

deal = deal.join(per_deal, on="DealID", how="left")
del per_deal
gc.collect()
# Il ciclo dei token mancanti che l'R applica a "deals" dopo il join.
deal = as_na(deal, R_NA_NAN)

In [ ]:
# ── 4.6 · la regola Zero_Invested ─────────────────────────────────────────
# I tipi di deal in cui l'importo manca in oltre il 90% dei casi ricevono 0
# invece di NA. E' una regola derivata da STATISTICHE GLOBALI su tutto il
# dataset - stessa famiglia della RandomForest, piu' mite - ma non e'
# un'imputazione modellistica: dice "questo tipo di deal non dichiara mai
# l'importo, quindi e' zero". Resta, ed e' da decidere se tenerla (voce M12).
per_tipo = (
    deal.group_by("DealType")
    .agg(pl.len().alias("n_totale"),
         pl.col("TotalInvestedCapital").is_null().sum().alias("n_mancanti"))
    # L'R costruisce la tabella con table() sui soli deal a importo mancante,
    # quindi i tipi che hanno SEMPRE l'importo non compaiono affatto.
    .filter(pl.col("n_mancanti") > 0)
    .with_columns((pl.col("n_mancanti") / pl.col("n_totale")).alias("quota_mancante"))
)
tipi_a_zero = per_tipo.filter(pl.col("quota_mancante") > 0.9)["DealType"].to_list()
# L'R costruiva anche un gruppo "Other" con i tipi rari, e poi DealTypeGrouped.
# Servivano solo come predittori della RandomForest: senza quella sono codice
# morto e non compaiono qui.
print(f"tipi a zero ({len(tipi_a_zero)}): {', '.join(sorted(tipi_a_zero))}")

deal = deal.with_columns(
    pl.when(pl.col("DealType").is_in(tipi_a_zero) & pl.col("TotalInvestedCapital").is_null())
    .then(0.0)
    .otherwise(pl.col("TotalInvestedCapital"))
    .alias("TotalInvestedCapital")
)
print(f"importi ancora mancanti: {deal['TotalInvestedCapital'].is_null().sum():,}")

In [ ]:
# ── 4.7 · i quattordici flag sul tipo di deal ─────────────────────────────
# Otto NON sono fra le 53 ma sono la cascata che produce GrowthStage, cioe' il
# target: Is_Preseed, Is_Seed, Is_EarlyVC, Is_LaterVC, Is_MA, Is_Public_Exit,
# Is_Out, Is_PE. Sei lo sono: Is_Debt, Is_Grant, Is_SpinOff, Is_CrowdFunding,
# Is_Accelerator, Is_Angel. Del panel completo manca solo Other_Deal.
PRESEED = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Spin-Off",
           "Equity Crowdfunding", "Product Crowdfunding", "Capitalization"]
USCITA_MA = ["Merger/Acquisition", "Buyout/LBO", "Debt - Acquisition", "Debt - Merger",
             "Merger of Equals", "Investor Buyout by Management", "Corporate Asset Purchase",
             "Reverse Merger"]
USCITA_PUBBLICA = ["IPO", "Secondary Transaction - Open Market",
                   "Secondary Transaction - Stock Distribution",
                   "Public Investment 2nd Offering", "PIPE"]
FALLIMENTI = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business",
              "Restart - Angel", "Restart - Early VC", "Restart - Later VC"]
DEBITO = ["Debt - General", "Debt Conversion", "Mezzanine", "Convertible Debt",
          "Debt Refinancing", "Debt - PPP", "Debt Repayment", "Dividend Recapitalization",
          "Exit Financing", "Project Financing", "Share Repurchase", "Leveraged Recapitalization"]
PRIVATE_EQUITY = ["PE Growth/Expansion", "Secondary Transaction - Private", "Corporate",
                  "Platform Creation", "GP Stakes", "General Corporate Purpose", "Capital Spending"]

FLAG_DEAL = {
    "Is_Preseed": PRESEED,
    "Is_Seed": ["Seed Round"],
    "Is_EarlyVC": ["Early Stage VC"],
    "Is_LaterVC": ["Later Stage VC"],
    "Is_MA": USCITA_MA,
    "Is_Public_Exit": USCITA_PUBBLICA,
    "Is_Out": FALLIMENTI,
    "Is_Debt": DEBITO,
    "Is_PE": PRIVATE_EQUITY,
    "Is_Grant": ["Grant"],
    "Is_SpinOff": ["Spin-Off"],
    "Is_CrowdFunding": ["Equity Crowdfunding", "Product Crowdfunding"],
    "Is_Accelerator": ["Accelerator/Incubator"],
    "Is_Angel": ["Angel (individual)"],
}
# is_in su un DealType nullo darebbe null; fill_null(False) lo rende falso,
# che e' quello che fa %in% in R.
deal = deal.with_columns(
    *[pl.col("DealType").is_in(v).fill_null(False).alias(f) for f, v in FLAG_DEAL.items()]
)

In [ ]:
# ── 4.8 · aggregare a (azienda, anno) ─────────────────────────────────────
importo = pl.col("TotalInvestedCapital")


def qualunque(colonna: str) -> pl.Expr:
    """any(x, na.rm = TRUE): un gruppo tutto mancante da' False, non nullo."""
    return pl.col(colonna).fill_null(False).any()


deals_panel = deal.group_by(["CompanyID", "Year_Delta"]).agg(
    pl.len().alias("N_Deal"),
    # sum(x, na.rm = TRUE): un importo ignoto vale ZERO. E' la semantica che
    # confonde "non ha raccolto niente" con "non sappiamo quanto"; il panel
    # completo produceva anche TotalRaised_NA e TotalRaised_any per distinguere
    # i due casi, ma nessuna delle due arriva alle 53 e la colonna scelta per il
    # modello e' questa.
    importo.fill_null(0.0).sum().alias("TotalRaised"),
    # I quattordici flag: acceso se ALMENO UN deal di quell'anno lo accende.
    *[qualunque(f).alias(f) for f in FLAG_DEAL if f not in ("Is_Accelerator", "Is_Angel")],
    # Queste due sono un OR fra il tipo di deal e la categoria dell'investitore:
    # un round da un acceleratore conta anche se il DealType non lo dice.
    (qualunque("Is_Accelerator") | qualunque("has_Accelerator")).alias("Is_Accelerator"),
    (qualunque("Is_Angel") | qualunque("has_Angel")).alias("Is_Angel"),
    # Quanti investitori nuovi in totale nell'anno: e' il peso delle medie
    # ponderate cumulate della fase 5.
    pl.col("TotalInvestors").fill_null(0).sum().alias("TotalInvestors"),
    pl.col("MeanTotalInvestments").mean().alias("MeanTotalInvestments"),
    pl.col("MeanMedianRoundAmount").mean().alias("MeanMedianRoundAmount"),
    # I quattro has_* sui nuovi investitori e i sei sui lead, tutti nelle 53.
    *[qualunque(f"has_{c}").alias(f"has_{c}")
      for c in ("Corporate", "VentureCapital", "PrivateEquity", "PublicInvestor")],
    *[qualunque(f"has_{c}_Lead").alias(f"has_{c}_Lead") for c in CATEGORIE],
    # tail(na.omit(CEOPBId), 1): l'ultimo CEO non nullo nell'ordine delle righe.
    tail_na_omit("CEOPBId").alias("CEO_ID"),
)
del deal
gc.collect()
print(f"deals_panel: {deals_panel.height:,} anni-azienda con almeno un deal")
print(f"  nel gruppo ad anno nullo (non si agganceranno): "
      f"{deals_panel.filter(pl.col('Year_Delta').is_null()).height:,}")

In [ ]:
# ── 4.9 · innestare nel panel, e TR_D ─────────────────────────────────────
panel = pl.read_parquet(cfg.interim("panel_team.parquet"))

panel = panel.join(deals_panel, on=["CompanyID", "Year_Delta"], how="left").with_columns(
    # TR_D = 1 quando quell'anno-azienda non ha avuto NESSUN deal.
    # Nel panel completo nasceva dal test "tutte e sei le varianti di
    # TotalRaised sono nulle". Con una variante sola il risultato e' identico -
    # verificato, zero divergenze su 1.001.625 righe - perche' TotalRaised e'
    # una sum(na.rm = TRUE) e quindi non e' MAI nulla dentro deals_panel:
    # diventa nulla solo quando il join non trova niente.
    pl.col("TotalRaised").is_null().cast(pl.Int64).alias("TR_D"),
)
del deals_panel
gc.collect()

# Il ciclo finale dei token mancanti dell'R, che include anche "Inf" e "-Inf":
# converte in NA i -Inf dei max() e i NaN dei mean() su gruppi tutti mancanti.
panel = as_na(panel, R_NA_INF)
panel.write_parquet(cfg.interim("panel_deals.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"TR_D = 1 (anno senza deal): {(panel['TR_D'] == 1).sum():,}")
del panel
gc.collect()

---
## Fase 5 — finalizzazione

I flag diventano **cumulativi** (una volta acceso, resta acceso), nasce
`GrowthStage`, e si agganciano i tre attributi del CEO.

Rispetto al panel completo spariscono `StageBlock` (che non è nemmeno
riproducibile e non la legge nessuno), `YearsInStage`, le due colonne di stadio
futuro non raggruppate, quindici dei diciotto attributi del CEO e quattro
delle sei cumulate.

In [ ]:
# ── 5.1 · i ventiquattro flag diventano cumulativi ────────────────────────
# cumany: un flag acceso in un anno resta acceso in tutti gli anni successivi.
# E' cosi' che "l'azienda ha gia' fatto un round seed" diventa uno stato e non
# un evento. Rende gli stadi monotoni: un'azienda non puo' retrocedere.
FLAG_CUMULATIVI = [
    "Is_Preseed", "Is_Seed", "Is_EarlyVC", "Is_LaterVC", "Is_MA", "Is_Public_Exit",
    "Is_Out", "Is_Debt", "Is_PE", "Is_Grant", "Is_SpinOff", "Is_CrowdFunding",
    "Is_Accelerator", "Is_Angel",
    "has_Corporate", "has_VentureCapital", "has_PrivateEquity", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
]
# has_Angel e has_Accelerator non sono in questa lista: sono gia' stati
# consumati dall'OR del blocco 4.8 e non servono piu'.

panel = pl.read_parquet(cfg.interim("panel_deals.parquet")).sort(["CompanyID", "Year_Delta"])
# .over("CompanyID") = il group_by dell'R: la cumulata riparte per ogni azienda.
panel = panel.with_columns(cumany(pl.col(c)).over("CompanyID").alias(c) for c in FLAG_CUMULATIVI)
print(f"flag resi cumulativi: {len(FLAG_CUMULATIVI)}")

In [ ]:
# ── 5.2 · GrowthStage, la cascata che definisce il target ─────────────────
stato = pl.col("OwnershipStatus")
out, pubblica, ma = pl.col("Is_Out"), pl.col("Is_Public_Exit"), pl.col("Is_MA")
later, pe = pl.col("Is_LaterVC"), pl.col("Is_PE")
early, seed, preseed = pl.col("Is_EarlyVC"), pl.col("Is_Seed"), pl.col("Is_Preseed")
non_terminale = ~ma & ~pubblica & ~out

# Sette condizioni a CORTO CIRCUITO: il primo match vince, l'ordine e' vincolante.
# Due fonti si mescolano qui, ed e' una scelta discutibile (voce M1):
#   - OwnershipStatus e' lo stato ATTUALE dell'azienda, agganciato al solo anno
#     della sua data (2.764 righe in tutto);
#   - i flag sono storici e cumulativi.
# Nei primi tre rami la prima fonte vince con un OR.
# Attenzione alla logica a tre valori: quando OwnershipStatus e' nullo,
# "null | True" vale True e il flag decide, mentre "null | False" vale null e la
# condizione non matcha - il ramo successivo viene provato. Identico in R.
panel = panel.with_columns(
    pl.when((stato == "Out of Business") | out).then(pl.lit("Out"))
    .when(stato.is_in(["Publicly Held", "In IPO Registration"]) | pubblica).then(pl.lit("Exit_Public"))
    .when(stato.is_in(["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]) | ma).then(pl.lit("Exit_M&A"))
    .when((later | pe) & non_terminale).then(pl.lit("LaterVC_or_Other"))
    .when(early & ~later & non_terminale & ~pe).then(pl.lit("EarlyVC"))
    .when(seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Seed"))
    .when(preseed & ~seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Preseed"))
    .otherwise(None)
    .alias("GrowthStage")
)
print(panel["GrowthStage"].value_counts(sort=True))

In [ ]:
# ── 5.3 · dove non c'e' nessun deal, il raccolto e' zero ──────────────────
# Corretto: un anno senza round non ha raccolto niente.
panel = panel.with_columns(
    pl.when(pl.col("TR_D") == 1).then(0.0).otherwise(pl.col("TotalRaised")).alias("TotalRaised")
)

# ── 5.4 · le cumulate ─────────────────────────────────────────────────────
# TotalInvestors arriva dalla fase 4 come "nuovi investitori di QUESTO anno".
# Va letto PRIMA di essere sovrascritto dalla propria cumulata, perche' e' il
# peso delle medie ponderate del blocco 5.5. Nell'R i due passaggi stanno nello
# stesso mutate e dplyr li valuta in sequenza; qui servono due with_columns
# separati, perche' polars valuta le espressioni di un singolo with_columns
# tutte sul frame di partenza.
panel = panel.with_columns(pl.col("TotalInvestors").fill_null(0).alias("NewInvestors"))

panel = panel.with_columns(
    # r_cum_sum invece di cum_sum: in R un NA avvelena tutto il resto del
    # vettore, cumsum(c(1, NA, 3)) fa c(1, NA, NA). Qui i fill_null(0) tolgono
    # ogni nullo, quindi le due versioni coinciderebbero - ma si usa la
    # primitiva giusta per non doverci ripensare se un giorno il fill sparisce.
    r_cum_sum(pl.col("N_Deal").fill_null(0)).over("CompanyID").alias("N_Deal"),
    r_cum_sum(pl.col("NewInvestors").fill_null(0)).over("CompanyID").alias("TotalInvestors"),
)
print(f"N_Deal massimo: {panel['N_Deal'].max()}   TotalInvestors massimo: {panel['TotalInvestors'].max()}")

In [ ]:
# ── 5.5 · le medie ponderate cumulate ─────────────────────────────────────
# "In media, quanto erano grandi gli investitori che hanno messo soldi in questa
# azienda fino a quest'anno", pesato per quanti erano.
# L'R lo calcola con un doppio ciclo, O(n^2). weighted_cumulative usa
# l'equivalente algebrico cumsum(x*w) / cumsum(w) sulle sole righe valide:
# O(n) ed ESATTO, non un'approssimazione.
PONDERATE = ["MeanTotalInvestments", "MeanMedianRoundAmount"]

panel = panel.with_columns(
    weighted_cumulative(c, "NewInvestors", ["CompanyID"]).alias(f"{c}_cum") for c in PONDERATE
)
# Le versioni non cumulate hanno finito: nelle 53 ci sono solo le _cum.
panel = panel.drop(*PONDERATE, "NewInvestors")

In [ ]:
# ── 5.6 · gli attributi del CEO ───────────────────────────────────────────
# Dei diciotto del panel completo ne restano TRE: sono gli unici nelle 53.
# (Highest_Degree_CEO e' selezionato ma poi scartato a valle dalla soglia del
# 40% di mancanti - e' nullo sul 64,7% del panel - e lo si tiene perche' la
# selezione delle feature avviene fuori di qui.)
CEO_VARS = ["Gender", "Highest_Degree", "WorkExperienceIndex"]

# CEO_ID e' dichiarato solo negli anni in cui c'e' stato un deal. Si propaga in
# avanti: chi era CEO all'ultimo round lo resta finche' non ne arriva un altro.
panel = panel.with_columns(pl.col("CEO_ID").fill_null(strategy="forward").over("CompanyID"))

ceo = (
    pl.read_parquet(cfg.interim("db3.parquet"))
    .select("CompanyID", "PersonID", *CEO_VARS)
    .rename({c: f"{c}_CEO" for c in CEO_VARS})
)
# Il join e' sulla COPPIA (azienda, persona): il CEO deve essere nel board team
# di quella stessa azienda. Misurato: lo e' nel 100% dei casi dichiarati.
panel = panel.join(
    ceo, left_on=["CompanyID", "CEO_ID"], right_on=["CompanyID", "PersonID"], how="left"
).drop("CEO_ID")
del ceo
gc.collect()
print(f"righe con attributi del CEO: {panel['Gender_CEO'].is_not_null().sum():,}")

In [ ]:
# ── 5.7 · le due colonne d'anagrafica, e la selezione ─────────────────────
# HQCountry e PrimaryIndustrySector sono due delle 53 e non sono mai entrate nel
# panel: vivono a livello azienda. Qui si agganciano, ed e' l'equivalente del
# db_final dell'R, che pero' ne portava ventidue.
panel = panel.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "HQCountry", "PrimaryIndustrySector"
    ),
    on="CompanyID",
    how="left",
)
# db_master_1 ha una riga per azienda, quindi il join non puo' moltiplicare le
# righe. Se lo facesse sarebbe un errore grave e silenzioso: meglio accorgersene.
righe_attese = panel.height

# Age e' l'eta' dell'azienda, cioe' Delta con un altro nome. E' una delle 53;
# Delta e Year_Delta restano come coordinate interne fino alla fine della fase 7.
panel = panel.with_columns(pl.col("Delta").alias("Age")).drop(
    # OwnershipStatus ha fatto il suo lavoro nella cascata GrowthStage.
    "OwnershipStatus", "Delta", "TR_D",
)
assert panel.height == righe_attese, "il join con l'anagrafica ha moltiplicato le righe"
panel.write_parquet(cfg.interim("panel_finale.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
del panel
gc.collect()

---
## Fase 6 — raggruppamento degli stadi e troncamento

Nessuno script R: il codice originale è andato perduto ed esisteva solo il suo
output. Le regole sono state ricostruite dal file e verificate contro di esso a
divergenza zero su tutte le 882.324 righe.

Qui non si costruisce `StageBlock` (non riproducibile, non letta da nessuno) né
`YearsInStage` (non è fra le 53).

In [ ]:
# ── 6.1 · dai sette stadi ai quattro gruppi ───────────────────────────────
GRUPPO_STADIO = {
    "Preseed": "Early",
    "Seed": "Early",
    "EarlyVC": "Early",
    "LaterVC_or_Other": "Later",
    "Out": "Out",
    "Exit_M&A": "Exit",
    "Exit_Public": "Exit",
}
GRUPPI_TERMINALI = ["Out", "Exit"]

panel = pl.read_parquet(cfg.interim("panel_finale.parquet")).sort(["CompanyID", "Year_Delta"])
# default=None: uno stadio nullo resta nullo, non diventa una categoria.
panel = panel.with_columns(
    pl.col("GrowthStage").replace_strict(GRUPPO_STADIO, default=None).alias("GrowthStageGroup")
).drop("GrowthStage")   # la versione a sette stadi non e' fra le 53
print(panel["GrowthStageGroup"].value_counts(sort=True))

In [ ]:
# ── 6.2 · lo stadio futuro, PRIMA del troncamento ─────────────────────────
# L'ordine conta: se si calcolasse dopo il troncamento, "Out" ed "Exit" non
# sarebbero piu' raggiungibili come stadio futuro - e il senso della colonna e'
# esattamente quello. E' l'errore piu' facile da fare riscrivendo questa fase.
panel = next_different(
    panel, "GrowthStageGroup", ["CompanyID"], "GrowthNextStageGroup", "TimeNextStageGroup"
)

# Quante righe restano all'azienda dopo questa: serve per le righe che non
# cambiano mai gruppo.
righe_rimanenti = pl.len().over("CompanyID") - pl.int_range(pl.len()).over("CompanyID") - 1
panel = panel.with_columns(
    # "Stay" NON e' un nullo: e' la stringa letterale che dice "questa azienda
    # non lascia il gruppo in cui e'". La fase 7 la sostituira' col gruppo corrente.
    pl.col("GrowthNextStageGroup").fill_null("Stay").alias("GrowthNextStageGroup"),
    # ...e la distanza e' quella dall'ULTIMA riga non troncata, non nulla.
    # Le due espressioni stanno nello stesso with_columns apposta: la seconda
    # legge GrowthNextStageGroup PRIMA che la prima lo riempia, perche' polars
    # valuta entrambe sul frame di partenza.
    pl.when(pl.col("GrowthNextStageGroup").is_null())
    .then(righe_rimanenti)
    .otherwise(pl.col("TimeNextStageGroup"))
    .alias("TimeNextStageGroup"),
)
print(panel["GrowthNextStageGroup"].value_counts(sort=True))

In [ ]:
# ── 6.3 · il troncamento all'uscita ───────────────────────────────────────
prima = panel.height
# cum_max su 0/1: vale 1 dalla prima riga terminale in poi. Si tengono solo le
# righe a 0, quindi anche la riga terminale stessa viene eliminata.
raggiunto_terminale = (
    pl.col("GrowthStageGroup")
    .is_in(GRUPPI_TERMINALI)
    .fill_null(False)
    .cast(pl.Int8)
    .cum_max()
    .over("CompanyID")
)
panel = panel.filter(raggiunto_terminale == 0)
# Nota: elimina anche le righe NON terminali che seguono una terminale - 5.364
# righe. Un'azienda che risulta "Out" e poi ha un altro deal viene troncata al
# primo Out e la sua vita successiva scompare. Coerente con l'idea che l'uscita
# sia assorbente, ma va detto (voce M18).
print(f"righe: {prima:,} -> {panel.height:,}  (attese 880.473)")
print(f"aziende: {panel['CompanyID'].n_unique():,}  (attese 116.313)")

---
## Fase 7 — competitor, anno per anno

Le tre colonne competitor vengono calcolate **due volte**: una versione
temporizzata (quanti concorrenti erano *vivi* in quell'anno) e una statica
(`*_All`). Entrambe sono fra le 53.

Nel panel completo questa fase sovrascriveva le sei colonne della fase 3a. Qui
la fase 3a non esiste: si calcolano e basta.

In [ ]:
# ── 7.1 · la finestra di vita di OGNI azienda ─────────────────────────────
# Si rilegge Company.csv per intero, non solo le coorti del panel: un
# concorrente puo' essere piu' vecchio del 2000 e va comunque considerato vivo.
COLONNE_VITA = [*COMPANY_DATE_COLUMNS, "FiscalPeriod", "YearFounded", "HQCountry"]
tutte = as_na(read_raw(cfg, "Company", ["CompanyID", *COLONNE_VITA]), R_NA_NAN)

trimestre_v = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
anno_v = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
tutte = tutte.with_columns(
    *[parse_date_r(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    pl.date(anno_v, trimestre_v * 3, 30).alias("FiscalDate"),
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

# Stessa definizione di MaxYear della fase 1: l'ultimo anno con DATI.
# Usarlo come "l'azienda era viva" e' il difetto metodologico principale di
# questa fase (voce M20): un'azienda ben coperta da PitchBook risulta viva piu'
# a lungo di una coperta male, quindi la temporizzazione sovrappesa
# sistematicamente i concorrenti grandi e ben documentati.
vita = (
    tutte.with_columns(
        pl.max_horizontal([pl.col(c).dt.year() for c in [*COMPANY_DATE_COLUMNS, "FiscalDate"]]).alias("MaxYear")
    )
    .select("CompanyID", "YearFounded", "MaxYear", "HQCountry")
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
)
del tutte
gc.collect()
print(f"aziende con una finestra di vita utilizzabile: {vita.height:,} su 134.355")

In [ ]:
# ── 7.2 · le coppie azienda / azienda simile ──────────────────────────────
simili_grezze = as_na(
    read_raw(cfg, "CompanySimilarRelation",
             ["CompanyID", "SimilarCompanyID", "SimilarityScore", "IsCompetitor"]),
    R_NA_NAN,
).with_columns(to_num("SimilarityScore"))

# Si tengono solo le coppie in cui la prima azienda e' nel panel e la seconda ha
# una finestra di vita calcolabile.
id_panel = panel.select("CompanyID").unique().to_series()
simili_grezze = simili_grezze.filter(
    # .implode() = "questa Serie e' un INSIEME di valori", non una colonna da
    # allineare riga per riga. Stesso risultato, senza il DeprecationWarning.
    pl.col("CompanyID").is_in(id_panel.implode())
    & pl.col("SimilarCompanyID").is_in(vita["CompanyID"].implode())
)
print(f"coppie utilizzabili: {simili_grezze.height:,}")

# La finestra del CONCORRENTE, rinominata per il range join del blocco 7.3.
finestra = vita.select(
    pl.col("CompanyID").alias("SimilarCompanyID"),
    pl.col("YearFounded").alias("YF"),
    pl.col("MaxYear").alias("MY"),
    pl.col("HQCountry").alias("_paese_concorrente"),
)

# Due insiemi diversi: "simili" (tutte le coppie, per la media di similarita')
# e "concorrenti" (solo IsCompetitor = "Yes", per i conteggi).
simili = simili_grezze.select("CompanyID", "SimilarCompanyID", "SimilarityScore").join(
    finestra.select("SimilarCompanyID", "YF", "MY"), on="SimilarCompanyID", how="inner"
)
concorrenti = (
    simili_grezze.filter(pl.col("IsCompetitor") == "Yes")
    .select("CompanyID", "SimilarCompanyID", "SimilarityScore")
    .join(finestra, on="SimilarCompanyID", how="inner")
    .join(vita.select("CompanyID", pl.col("HQCountry").alias("_paese_proprio")),
          on="CompanyID", how="left")
    .with_columns(
        # Nullo se manca uno dei due paesi: la coppia viene esclusa dal
        # conteggio, non contata come "paese diverso".
        (pl.col("_paese_proprio") == pl.col("_paese_concorrente")).alias("_stesso_paese")
    )
    .drop("_paese_proprio", "_paese_concorrente")
)
del simili_grezze, finestra, vita
gc.collect()
print(f"coppie concorrenti: {concorrenti.height:,}   coppie simili: {simili.height:,}")

In [ ]:
# ── 7.3 · chi era attivo in quale anno ────────────────────────────────────
anni_panel = panel.select("CompanyID", "Year_Delta").unique()

# active_pairs e' un RANGE JOIN: per ogni (azienda, anno) del panel tiene le
# coppie il cui concorrente era vivo quell'anno, cioe' YF <= anno <= MY.
# join_where lo esegue come join di disuguaglianza invece di materializzare il
# prodotto cartesiano: e' l'unico modo perche' ci stia in memoria, il risultato
# e' gia' di diversi milioni di righe.
attivi_concorrenti = active_pairs(anni_panel, concorrenti, ["_stesso_paese"])
attivi_simili = active_pairs(anni_panel, simili, ["SimilarityScore"])
print(f"coppie concorrente-anno attive: {attivi_concorrenti.height:,}")
print(f"coppie simile-anno attive     : {attivi_simili.height:,}")

In [ ]:
# ── 7.4 · tre colonne temporizzate e tre statiche ─────────────────────────
stat_concorrenti = attivi_concorrenti.group_by("CompanyID", "Year_Delta").agg(
    pl.col("SimilarCompanyID").n_unique().alias("N_Competitors"),
    # Attenzione: Same_Country cambia SIGNIFICATO mantenendo il nome. Alla fase
    # 3a del panel completo era un booleano - "esiste un concorrente sopra 90 di
    # similarita' nel nostro paese" - e qui diventa un CONTEGGIO di concorrenti
    # attivi nello stesso paese. La feature dei modelli si chiama uguale.
    pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country"),
)
stat_simili = attivi_simili.group_by("CompanyID", "Year_Delta").agg(
    pl.col("SimilarityScore").mean().alias("SimilarityScoreMean")
)
del attivi_concorrenti, attivi_simili
gc.collect()

# Le stesse aggregazioni SENZA filtro temporale: sono le versioni _All, usate
# dall'esperimento senza finestra temporale.
statiche = concorrenti.group_by("CompanyID").agg(
    pl.col("SimilarCompanyID").n_unique().alias("N_Competitors_All"),
    pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country_All"),
)
statiche_simili = simili.group_by("CompanyID").agg(
    pl.col("SimilarityScore").mean().alias("SimilarityScoreMean_All")
)
del concorrenti, simili
gc.collect()
print(f"anni-azienda con almeno un concorrente attivo: {stat_concorrenti.height:,}")

In [ ]:
# ── 7.5 · innestare, sostituire "Stay", rinumerare ────────────────────────
finale = (
    panel
    .join(stat_concorrenti, on=["CompanyID", "Year_Delta"], how="left")
    .join(stat_simili, on=["CompanyID", "Year_Delta"], how="left")
    .join(statiche, on="CompanyID", how="left")
    .join(statiche_simili, on="CompanyID", how="left")
    .with_columns(
        # Nessun concorrente attivo -> zero concorrenti. Corretto.
        pl.col("N_Competitors", "Same_Country", "N_Competitors_All", "Same_Country_All").fill_null(0),
        # Nessun concorrente attivo -> similarita' media 0. Discutibile: zero e'
        # il MINIMO della scala, non un valore neutro, quindi un'azienda senza
        # concorrenti vivi appare a un modello come un'azienda i cui concorrenti
        # sono massimamente diversi (voce M22).
        pl.col("SimilarityScoreMean", "SimilarityScoreMean_All").fill_null(0.0),
    )
    .with_columns(
        # "Stay" significa che l'azienda non lascia il gruppo in cui e': lo
        # stadio futuro e' quello corrente.
        pl.when(pl.col("GrowthNextStageGroup") == "Stay")
        .then(pl.col("GrowthStageGroup"))
        .otherwise(pl.col("GrowthNextStageGroup"))
        .alias("GrowthNextStageGroup")
    )
)
del panel, stat_concorrenti, stat_simili, statiche, statiche_simili
gc.collect()

# La rinumerazione e' l'ULTIMA operazione della pipeline, e non per caso:
# distrugge ogni possibilita' di join con i file di riferimento.
mappa_id = finale.select("CompanyID").unique().sort("CompanyID").with_row_index("_nuovo", offset=1)
finale = finale.join(mappa_id, on="CompanyID", how="left").drop("CompanyID").rename({"_nuovo": "CompanyID"})

# Le 53 colonne di example_panel, nello stesso ordine, con TotalRaised al posto
# di TotalRaised_Est. Year_Delta ha fatto da chiave fino a qui e si ferma.
COLONNE_FINALI = [
    "CompanyID", "Age", "YearFounded",
    "GrowthStageGroup", "GrowthNextStageGroup", "TimeNextStageGroup",
    "N_Deal", "TotalRaised", "Percent_Females",
    "Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT",
    "Institute", "WorkExp_Idx_Mean", "Total_Founders",
    "Is_Debt", "Is_SpinOff", "Is_CrowdFunding", "MeanMedianRoundAmount_cum",
    "Is_Accelerator", "has_Corporate", "has_VentureCapital", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
    "HQCountry", "PrimaryIndustrySector",
    "SimilarityScoreMean", "N_Competitors", "Same_Country",
    "Highest_Degree_CEO", "Gender_CEO", "MeanTotalInvestments_cum", "WorkExperienceIndex_CEO",
    "Is_Angel", "Total_People", "Is_Grant", "has_PrivateEquity", "TotalInvestors",
    "Highest_Degree_Mean", "Avg_Earliest_Year",
    "N_Competitors_All", "Same_Country_All", "SimilarityScoreMean_All",
]
finale = finale.select(COLONNE_FINALI)

finale.write_parquet(cfg.interim("panel.parquet"))
finale.write_csv(cfg.interim("panel.csv.gz"), compression="gzip")
print(f"panel finale: {finale.height:,} righe x {finale.width} colonne")
print(f"'Stay' residui: {(finale['GrowthNextStageGroup'] == 'Stay').sum()}   (deve essere 0)")

---
## Verifica

L'equivalenza col panel completo è già stata dimostrata e registrata in
`docs/panel_revisione_stato.md`: prima a correzioni spente (882.324 righe, zero
divergenze), poi con le due correzioni accese su entrambi i notebook (880.473
righe, zero divergenze). Non serve rifarla a ogni esecuzione — servirebbe
rieseguire il notebook completo, che a correzioni accese non sta in memoria in
un processo solo.

Quello che serve a ogni esecuzione è accorgersi se qualcosa si è mosso. Questa
cella controlla le invarianti misurate allora: sono legate a **questa
estrazione**, quindi un `assert` che salta significa «è cambiato qualcosa,
capisci cosa prima di usare il risultato», non necessariamente «c'è un bug».

In [ ]:
# ── Verifica delle invarianti ─────────────────────────────────────────────
finale = pl.read_parquet(cfg.interim("panel.parquet"))

ATTESI = {
    "righe": 880_473,
    "aziende": 116_313,
    "colonne": 53,
    "righe con dati di team": 816_297,
    "righe con stadio": 637_043,
}
ottenuti = {
    "righe": finale.height,
    "aziende": finale["CompanyID"].n_unique(),
    "colonne": finale.width,
    "righe con dati di team": int(finale["Total_People"].is_not_null().sum()),
    "righe con stadio": int(finale["GrowthStageGroup"].is_not_null().sum()),
}
for nome, atteso in ATTESI.items():
    ott = ottenuti[nome]
    stato = "ok" if ott == atteso else f"ATTESO {atteso:,}"
    print(f"  {nome:<24}{ott:>10,}   {stato}")

# Le due correzioni permanenti devono essere visibili nel risultato.
assert finale.filter(pl.col("Age") < 0).height == 0, "B6: ci sono ancora anni prima della fondazione"
coorte2000 = finale.filter(pl.col("YearFounded") == 2000)
quota = 100 * coorte2000["Total_People"].is_not_null().sum() / coorte2000.height
assert quota > 80, f"B1: la coorte 2000 ha solo il {quota:.1f}% di righe con team"
print(f"\n  B6: righe con Age < 0        0   ok")
print(f"  B1: coorte 2000 con team  {quota:5.1f}%  ok  (era 0,0%)")

# Le colonne devono essere esattamente quelle di example_panel, con
# TotalRaised al posto di TotalRaised_Est.
esempio = pl.read_csv("data/raw/example_panel.csv", infer_schema_length=0).columns
atteso_colonne = [c if c != "TotalRaised_Est" else "TotalRaised" for c in esempio]
assert finale.columns == atteso_colonne, "le colonne non coincidono con example_panel"
print("  colonne identiche a example_panel   ok")

if all(ottenuti[k] == v for k, v in ATTESI.items()):
    print("\nTUTTE LE INVARIANTI RISPETTATE.")
else:
    print("\nQUALCOSA E' CAMBIATO: confronta con docs/panel_revisione_stato.md.")